# OOP in Python for Deep Learning

This notebook teaches Object-Oriented Programming (OOP) in Python from a deep-learning engineer perspective.

You know basic Python already; this guide focuses on how OOP helps you design scalable model, data, and training systems.

## Learning Path

1. Classes and Objects
2. Constructors (`__init__`)
3. Instance vs Class Variables
4. Methods (instance, class, static)
5. Encapsulation
6. Inheritance
7. Method Overriding
8. Multiple Inheritance
9. Abstraction
10. Magic Methods (`__str__`, `__len__`, etc.)
11. Composition vs Inheritance
12. Designing Modular Deep Learning Systems with OOP

After every 3 topics, you get a mini project. At the end, you get a final project: a small OOP deep-learning framework.


## 1) Classes and Objects

### 1. Concept Explanation

A class is a blueprint. An object is an instance created from that blueprint.

In deep learning, classes let you represent real components as objects: models, datasets, optimizers, trainers, and callbacks. This keeps code modular and testable.

### 4. Code Walkthrough

- `class LayerConfig:` creates a custom type.
- `cfg = LayerConfig(...)` creates an object with its own state.
- `class TinyNet(nn.Module):` defines a model blueprint.
- `net = TinyNet(...)` is a model object that can do forward passes.

### 5. Practice Exercise

Create a `TokenizerConfig` class with `vocab_size` and `max_len`. Instantiate 2 different objects and print their attributes.

### 6. Challenge Exercise

Create a `SimpleClassifier` class (inherits `nn.Module`) with one hidden layer and ReLU. Instantiate it for `input_dim=20, hidden_dim=32, num_classes=4`.


In [ ]:
# 2. Simple Python Example
class LayerConfig:
    def __init__(self, in_features, out_features):
        self.in_features = in_features
        self.out_features = out_features

cfg = LayerConfig(128, 64)
print('Simple object:', cfg.in_features, '->', cfg.out_features)

# 3. Deep Learning Application
import torch
import torch.nn as nn

class TinyNet(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.fc = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.fc(x)

net = TinyNet(10, 2)
x = torch.randn(4, 10)
print('Model output shape:', net(x).shape)

## 2) Constructors (`__init__`)

### 1. Concept Explanation

`__init__` runs when an object is created. It initializes object state.

In deep learning, constructors define architecture pieces and config: layers, loss, learning rate, file paths, transforms.

### 4. Code Walkthrough

- Constructor parameters become object attributes via `self.*`.
- In `ImageDataset`, constructor stores paths and labels for later indexing.
- In `MLP`, constructor wires linear layers once, not per forward pass.

### 5. Practice Exercise

Write a class `ExperimentConfig` with `batch_size`, `lr`, `epochs` set in `__init__`.

### 6. Challenge Exercise

Create a `RegressionDataset` class with constructor inputs `X` and `y`, and implement `__len__` and `__getitem__`.


In [ ]:
# 2. Simple Python Example
class ExperimentConfig:
    def __init__(self, batch_size, lr):
        self.batch_size = batch_size
        self.lr = lr

cfg = ExperimentConfig(32, 1e-3)
print(cfg.batch_size, cfg.lr)

# 3. Deep Learning Application
from torch.utils.data import Dataset

class ImageDataset(Dataset):
    def __init__(self, image_paths, labels):
        self.image_paths = image_paths
        self.labels = labels

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Placeholder loading logic for teaching
        x = torch.randn(3, 32, 32)
        y = self.labels[idx]
        return x, y

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

## 3) Instance vs Class Variables

### 1. Concept Explanation

Instance variables belong to each object (`self.var`). Class variables are shared across all objects of a class.

In deep learning, shared metadata (e.g., default device, version, global counters) can be class variables, while run-specific settings should be instance variables.

### 4. Code Walkthrough

- `ModelTracker.total_models` is shared by all instances.
- `self.name` and `self.params` differ per instance.
- `BaseDataset.default_dtype` acts as a global default.

### 5. Practice Exercise

Create a class `Run` with class variable `run_count=0`; increment it in `__init__`.

### 6. Challenge Exercise

Create a model wrapper class with class variable `default_device='cpu'` and instance variable `device` that can override it.


In [ ]:
# 2. Simple Python Example
class ModelTracker:
    total_models = 0  # class variable

    def __init__(self, name, params):
        self.name = name
        self.params = params
        ModelTracker.total_models += 1

m1 = ModelTracker('A', 1000)
m2 = ModelTracker('B', 2000)
print(m1.name, m2.name, ModelTracker.total_models)

# 3. Deep Learning Application
class BaseDataset(Dataset):
    default_dtype = torch.float32  # shared across all datasets

    def __init__(self, n_samples):
        self.n_samples = n_samples

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        x = torch.randn(10, dtype=self.default_dtype)
        y = torch.tensor(idx % 2)
        return x, y

## Mini Project 1 (Topics 1-3)

Build a small classification scaffold:

1. Create `DataConfig`, `ModelConfig`, `TrainConfig` classes.
2. Add instance variables for run-specific values (batch size, lr).
3. Add one class variable in `ModelConfig` to track how many model configs were created.
4. Instantiate configs and print a compact run summary.


## 4) Methods (Instance, Class, Static)

### 1. Concept Explanation

- Instance method: needs object state (`self`).
- Class method: works with class state (`cls`).
- Static method: utility logic, no `self` or `cls`.

In deep learning projects, this split keeps APIs clean: model behavior as instance methods, constructors/helpers as class methods, pure utilities as static methods.

### 4. Code Walkthrough

- `num_parameters` reads `self.model`.
- `from_small_preset` returns an instance using `cls(...)`.
- `accuracy` is stateless math.

### 5. Practice Exercise

Implement a class `Scaler` with an instance method `transform`, class method `from_data`, and static method `clip`.

### 6. Challenge Exercise

Add to a trainer class: class method `from_config(dict_cfg)` and static method `seed_everything(seed)`.


In [ ]:
class TrainerUtils:
    def __init__(self, model):
        self.model = model

    def num_parameters(self):  # instance method
        return sum(p.numel() for p in self.model.parameters())

    @classmethod
    def from_small_preset(cls):  # class method
        model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
        return cls(model)

    @staticmethod
    def accuracy(logits, y):  # static method
        pred = logits.argmax(dim=1)
        return (pred == y).float().mean().item()

utils = TrainerUtils.from_small_preset()
print('params:', utils.num_parameters())

## 5) Encapsulation

### 1. Concept Explanation

Encapsulation means bundling data and behavior together, and controlling access to internal state.

In deep learning, this prevents accidental changes to critical training state (learning rate, best metric, checkpoints).

### 4. Code Walkthrough

- `_lr` is treated as internal state by convention.
- `@property` exposes read access as `optimizer.lr`.
- Setter validates input before changing `_lr`.

### 5. Practice Exercise

Create a class `EarlyStoppingState` with private-ish `_best_loss`, and methods to update/read it safely.

### 6. Challenge Exercise

Build a `CheckpointManager` class that allows saving only if validation loss improves.


In [ ]:
class OptimizerConfig:
    def __init__(self, lr):
        self._lr = lr

    @property
    def lr(self):
        return self._lr

    @lr.setter
    def lr(self, value):
        if value <= 0:
            raise ValueError('Learning rate must be positive')
        self._lr = value

cfg = OptimizerConfig(1e-3)
cfg.lr = 5e-4
print('validated lr:', cfg.lr)

## 6) Inheritance

### 1. Concept Explanation

Inheritance lets a child class reuse and extend behavior from a parent class.

In deep learning, inheritance is everywhere: `nn.Module`, `Dataset`, callback systems, and trainer extensions.

### 4. Code Walkthrough

- `BaseTrainer` provides common loop structure.
- `ClassifierTrainer` inherits it and adds task-specific loss logic.

### 5. Practice Exercise

Create `BaseMetric` class and child classes `AccuracyMetric` and `MAEMetric`.

### 6. Challenge Exercise

Create `BaseDataset` with shared transform support, then derive `ImageDataset` and `TextDataset`.


In [ ]:
class BaseTrainer:
    def __init__(self, model):
        self.model = model

    def train_step(self, x, y):
        raise NotImplementedError

class ClassifierTrainer(BaseTrainer):
    def __init__(self, model, criterion):
        super().__init__(model)
        self.criterion = criterion

    def train_step(self, x, y):
        logits = self.model(x)
        return self.criterion(logits, y)

model = nn.Linear(10, 3)
trainer = ClassifierTrainer(model, nn.CrossEntropyLoss())
print('Inherited trainer ready:', isinstance(trainer, BaseTrainer))

## Mini Project 2 (Topics 4-6)

Build an extensible trainer skeleton:

1. `BaseTrainer` with `fit`, `train_epoch`, and abstract-ish `compute_loss`.
2. `ClassificationTrainer` child implementing cross-entropy loss.
3. Add encapsulated learning-rate property with validation.
4. Add one class method to build trainer from a config dictionary.


## 7) Method Overriding

### 1. Concept Explanation

Method overriding means redefining a parent method in a child class with task-specific behavior.

In deep learning, the same training skeleton can support classification, regression, contrastive learning, etc. by overriding one method.

### 4. Code Walkthrough

- Parent class defines `compute_loss`.
- Child classes override with different losses.

### 5. Practice Exercise

Add `RegressionTrainer` overriding `compute_loss` with `nn.MSELoss`.

### 6. Challenge Exercise

Add `LabelSmoothingTrainer` overriding classification loss with label smoothing.


In [ ]:
class GenericTrainer:
    def __init__(self, model):
        self.model = model

    def compute_loss(self, pred, target):
        raise NotImplementedError

class CETrainer(GenericTrainer):
    def __init__(self, model):
        super().__init__(model)
        self.criterion = nn.CrossEntropyLoss()

    def compute_loss(self, pred, target):
        return self.criterion(pred, target)

class MSETrainer(GenericTrainer):
    def __init__(self, model):
        super().__init__(model)
        self.criterion = nn.MSELoss()

    def compute_loss(self, pred, target):
        return self.criterion(pred, target)

## 8) Multiple Inheritance

### 1. Concept Explanation

Multiple inheritance means inheriting from more than one parent class.

In deep learning systems, this is useful for mixing capabilities (logging + checkpointing + early stopping), but should be used carefully to avoid complexity.

### 4. Code Walkthrough

- `LoggingMixin` and `CheckpointMixin` each provide one concern.
- `AdvancedTrainer` combines both in one class.

### 5. Practice Exercise

Add an `EarlyStoppingMixin` and combine it with `AdvancedTrainer`.

### 6. Challenge Exercise

Use `super()` consistently across mixins and inspect method resolution order (MRO).


In [ ]:
class LoggingMixin:
    def log(self, msg):
        print(f'[LOG] {msg}')

class CheckpointMixin:
    def save_checkpoint(self, path):
        print(f'Saving checkpoint to {path}')

class AdvancedTrainer(LoggingMixin, CheckpointMixin):
    def train_one_epoch(self):
        self.log('training epoch...')

adv = AdvancedTrainer()
adv.train_one_epoch()
adv.save_checkpoint('model.pt')

## 9) Abstraction

### 1. Concept Explanation

Abstraction defines what an object must do, without forcing one concrete implementation.

In deep learning, abstractions give interchangeable components: different losses, schedulers, data sources, or trainers with the same interface.

### 4. Code Walkthrough

- `AbstractDataset` declares required methods.
- Child classes must implement those methods.

### 5. Practice Exercise

Create abstract class `BaseAugmentation` with method `apply(x)`, then implement `GaussianNoiseAug`.

### 6. Challenge Exercise

Create abstract class `BaseTrainer` with `train_step` and `validate_step`, then implement a concrete `VisionTrainer`.


In [ ]:
from abc import ABC, abstractmethod

class AbstractDataset(ABC):
    @abstractmethod
    def __len__(self):
        pass

    @abstractmethod
    def __getitem__(self, idx):
        pass

class RandomVectorDataset(AbstractDataset):
    def __init__(self, n):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        return torch.randn(10), torch.tensor(idx % 2)

## Mini Project 3 (Topics 7-9)

Build a pluggable training architecture:

1. Abstract `BaseTrainer` with `compute_loss`.
2. Two overridden trainers: classification and regression.
3. Add `LoggingMixin` and `CheckpointMixin`.
4. Train on synthetic data to verify each trainer works.


## 10) Magic Methods (`__str__`, `__len__`, etc.)

### 1. Concept Explanation

Magic methods integrate your objects with Python language features.

In deep learning, they improve debuggability and ergonomics (printing config, dataset length, object representation).

### 4. Code Walkthrough

- `__str__` makes readable print output.
- `__len__` enables built-in `len(obj)`.
- `__repr__` helps in debugging and notebooks.

### 5. Practice Exercise

Add `__repr__` to your config class to show all fields in one line.

### 6. Challenge Exercise

Implement a dataset wrapper with `__len__`, `__getitem__`, and `__iter__` for quick iteration.


In [ ]:
class TrainingRun:
    def __init__(self, name, num_batches):
        self.name = name
        self.num_batches = num_batches

    def __str__(self):
        return f'TrainingRun(name={self.name}, batches={self.num_batches})'

    def __len__(self):
        return self.num_batches

run = TrainingRun('baseline', 120)
print(run)
print('len(run):', len(run))

## 11) Composition vs Inheritance

### 1. Concept Explanation

- Inheritance: "is-a" relationship.
- Composition: "has-a" relationship.

In deep learning systems, composition is often safer and more flexible: a trainer has a model, optimizer, scheduler, and logger.

### 4. Code Walkthrough

- `Trainer` receives dependencies in constructor (composition).
- You can swap optimizer/scheduler without changing trainer class hierarchy.

### 5. Practice Exercise

Build a `Trainer` that composes `model`, `optimizer`, and `criterion` only.

### 6. Challenge Exercise

Add a composed `CallbackManager` that runs hooks at epoch start/end.


In [ ]:
class MetricTracker:
    def update(self, loss):
        print(f'loss={loss:.4f}')

class ComposedTrainer:
    def __init__(self, model, optimizer, metric_tracker):
        self.model = model
        self.optimizer = optimizer
        self.metric_tracker = metric_tracker

    def log_loss(self, loss):
        self.metric_tracker.update(loss)

model = nn.Linear(10, 2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
trainer = ComposedTrainer(model, optimizer, MetricTracker())
trainer.log_loss(0.42)

## 12) Designing Modular Deep Learning Systems with OOP

### 1. Concept Explanation

A production-ready deep learning codebase uses OOP to separate concerns and enable extension.

Recommended modules:

- `configs`: experiment settings
- `data`: dataset and dataloader builders
- `models`: model classes
- `training`: trainer, loops, checkpointing
- `evaluation`: metrics

### 4. Code Walkthrough

- `ModelFactory` builds models from config names.
- `Trainer` composes model/optimizer/criterion.
- `fit` handles orchestration only.

### 5. Practice Exercise

Implement `ModelFactory.create(name)` for `mlp` and `linear`.

### 6. Challenge Exercise

Implement a `Pipeline` class that loads data, builds model from config, trains, evaluates, and saves best model.


In [ ]:
class ModelFactory:
    @staticmethod
    def create(name, in_dim=10, out_dim=2):
        if name == 'linear':
            return nn.Linear(in_dim, out_dim)
        if name == 'mlp':
            return nn.Sequential(nn.Linear(in_dim, 32), nn.ReLU(), nn.Linear(32, out_dim))
        raise ValueError(f'Unknown model: {name}')

class SimplePipeline:
    def __init__(self, model_name):
        self.model = ModelFactory.create(model_name)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        self.criterion = nn.CrossEntropyLoss()

    def train_step(self, x, y):
        self.optimizer.zero_grad()
        logits = self.model(x)
        loss = self.criterion(logits, y)
        loss.backward()
        self.optimizer.step()
        return loss.item()

## Mini Project 4 (Topics 10-12)

Build a modular image classification pipeline:

1. Add useful magic methods to your config and run tracker classes.
2. Use composition for trainer dependencies (`model`, `optimizer`, `criterion`, `metrics`).
3. Build a simple model factory.
4. Train for a few epochs on synthetic data and print per-epoch metrics.


# Final Project: Build a Small OOP Deep Learning Training Framework

## Goal

Create a reusable framework with clear OOP boundaries.

## Required Components

1. `ExperimentConfig` class
2. `Dataset` class (synthetic or real)
3. `Model` class inheriting `nn.Module`
4. `Trainer` base class + one specialized trainer (override loss/step)
5. `MetricTracker` class
6. `CheckpointManager` class
7. `Pipeline` or `Runner` class to orchestrate training

## Suggested File Structure

- `configs.py`
- `data.py`
- `models.py`
- `trainer.py`
- `metrics.py`
- `checkpoint.py`
- `main.py`

## Stretch Goals

- Add mixins for logging and early stopping.
- Add abstraction via ABC for trainer interface.
- Add magic methods for better debug output.
- Add factory method to create models from config names.

## Self-Evaluation Checklist

- Did you use constructors to initialize all required state?
- Did you separate shared vs per-instance data correctly?
- Did you use overriding where task behavior differs?
- Did you prefer composition where it improved flexibility?
- Can you swap model or optimizer with minimal code changes?

When you finish, run one short experiment and print train/validation metrics for at least 3 epochs.
